# Tutorial 8: Three-Model Coupling

Estimated time: 25-45 minutes

## Prerequisites
No extra dependencies beyond base package runtime — everything here is numpy arithmetic
over a spec. Do T7 first: this tutorial assumes you know what a coupling asserts. (If you
are working through the whole series in one kernel, `environment-all.yml` builds
`py312_bayesmm_all`, which carries both optional backends.)

## Learning aims
- Primary package aim: extend the coupling graph to a third model and propagate along a *chain* `y → C → z → w`.
- Secondary scientific aim: budget uncertainty across that chain — a *deterministic* coupling adds no width, a *probabilistic* one adds width in quadrature — and see, concretely, why the default `--method propagate` is forward propagation and not inference.

## Success criteria
- you can read `std(y)`, `std(z)`, `std(w)` off the samples, say what the spec's σ values predict for each, and judge the agreement in units of Monte Carlo error rather than by eye;
- you can name one thing `propagate` does that sampling a joint posterior would never do.

## Why this tutorial matters

T7 coupled two models: `y` and `C` were pushed into agreement by a single probabilistic
link. Three models give you a **chain**: `y → C → z → w`. The σ values along that chain
decide how much uncertainty accumulates as you walk downstream — and the coupling *kinds*
decide whether anything accumulates at all.

Two ideas carry the whole notebook:

- **Widths add in quadrature, not linearly.** A `gaussian_link` of width σ turns `std = s`
  into `std = sqrt(s² + σ²)`. *Variances* add; standard deviations do not. That one fact is
  why a σ well below the prior width is nearly invisible, and why a noise budget is kept in
  variance units.
- **A `deterministic` coupling is free.** `target = transform(source)` exactly — no new
  randomness, so no new width. The step is invisible in the budget. It transmits
  uncertainty; it does not create any.

With the σ values this spec ships with, the two probabilistic links happen to contribute
exactly as much variance as the prior itself — so `std(w)` should land near `√2 × std(y)`,
i.e. the variance doubles along the chain. Step 2 prints observed and predicted side by
side *with an error bar*, so you can check that instead of taking it on faith.

One warning before you start, because it governs how to read every number below:
`bayesmm meta sample` runs `--method propagate` by default, and propagation is **not**
inference. We come back to what that means — and to a way of *seeing* it — right after the
run.

## Step 1: Read the spec, then build and sample

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)


In [ ]:
import json
from pathlib import Path

from bayesian_metamodeling.spec import MetaModelSpec

SPEC_REL = "tutorials/specs/metamodel.three_model.pymc.json"
SPEC_PATH = root / SPEC_REL
spec = MetaModelSpec.model_validate_json(SPEC_PATH.read_text())

# Print the chain in generative form, straight from the spec. Nothing here is typed by
# hand, so this stays true after you edit a σ in the checkpoint exercise below.
declared = {p.variable: p.distribution for p in spec.priors}
print(f"{SPEC_REL}\n")
print("Priors declared in the spec:")
for name, dist in declared.items():
    print(f"  {name} ~ Normal(loc={dist.get('loc')}, scale={dist.get('scale')})")

undeclared = [v.name for v in spec.variables if v.name not in declared]
print(f"\nVariables with NO declared prior: {undeclared}")
print("  -> the sampler falls back to Normal(0, 1) for these (`meta/sampling.py::_prior_params`).")
print("     They are coupling targets, so the fallback draw is normally overwritten.")
print("     Normally. Step 3 is about the case where it is not.")

print("\nCouplings, in the order the spec lists them (that order matters — Step 3):")
for i, c in enumerate(spec.couplings):
    transform = c.transform or {"kind": "identity"}
    if transform.get("kind") == "affine":
        rhs = f"{transform.get('alpha', 1.0)}*{c.source} + {transform.get('beta', 0.0)}"
    else:
        rhs = c.source
    if c.kind == "deterministic":
        note = "[deterministic — exact, adds NO width]"
    else:
        rhs = f"{rhs} + Normal(0, {c.sigma})"
        note = f"[{c.kind} — adds width σ={c.sigma}]"
    print(f"  {i}. {c.target} = {rhs:<24} {note}")

print(f"\nSurrogates referenced: {len(spec.surrogate_refs)}")
for ref in spec.surrogate_refs:
    print(f"  {ref}")

**Predict before you sample.** The printout above is the entire generative story:

- `y` starts at its prior, `Normal(0, 1)`.
- `y → C` is probabilistic: `C = y + Normal(0, σ₁)`.
- `C → z` is deterministic: `z = C` exactly. **No new noise.**
- `z → w` is probabilistic: `w = z + Normal(0, σ₂)`.

Write down three answers before running anything below:

1. Rank `std(y)`, `std(C)`, `std(z)`, `std(w)` widest to narrowest. Are any two of them
   *exactly* equal, and if so why?
2. Quantitatively: with independent noises adding in quadrature, what are `std(z)` and
   `std(w)` in terms of `std(y)`, σ₁ and σ₂? Put the printed σ values in and get numbers.
3. `corr(z, w)` — nearer 1.0 or nearer 0.5? (Hint: `w = z + noise`, so the correlation is
   `std(z) / std(w)`. A link that widens a variable also decorrelates it from its source.)

Step 2 prints all four widths and that correlation. Hold your answers until then.

In [ ]:
# 4000 draws x 2 chains = 8000 samples. The Monte Carlo error on a std falls as
# std/sqrt(2N), so this puts it under 1% — the effect we are about to measure has to be
# bigger than the error bar on the measurement, or there is nothing to read.
_ = run_mm_cli('meta', 'build', SPEC_REL)
_ = run_mm_cli(
    'meta', 'sample', SPEC_REL,
    '--draws', '4000', '--tune', '100', '--chains', '2', '--seed', '321',
)

### What `--method propagate` just did — and what it did not

`meta sample` has two methods and the default is the cheap one. Under `propagate`
(`meta/sampling.py::_sample_core`) the sampler:

1. draws **every** variable independently from its prior, then
2. walks the coupling factors **in spec order**, overwriting each coupling's *target* with
   `transform(source)` — plus Gaussian noise when the link is probabilistic.

Three consequences to hold on to:

- **Information flows one way.** A coupling reshapes its *target* and leaves its *source*
  at exactly its prior. In the samples you just produced, `y` is untouched `Normal(0, 1)`;
  it learns nothing from `C`, `z` or `w`. A posterior moves both ends of a coupling. So
  "posterior" is the wrong word for what we are about to plot. The right words are
  **forward uncertainty propagation**.
- **The surrogates are not consulted.** The compiled density is evaluated with
  `surrogates={}`, so every surrogate likelihood factor contributes exactly zero
  (`meta/compiler.py`: `if surrogate is None: continue`). This spec names three surrogates
  and all three are inert here. What you are watching is the coupling graph, alone.
- **`--tune 100` and `--chains 2` are recorded but inert.** There is no chain to burn in and
  no convergence to assess; `chains` merely reshapes one RNG stream into a
  `(chains, draws)` array, which is why Step 2 can flatten it with `.reshape(-1)` without a
  second thought. Under `--method joint` both flags start to matter, `inference_data.json`
  gains an `accept_rate`, and flattening would hide non-convergence.

`--method joint` (`meta/joint_sampling.py`) runs random-walk Metropolis over the *full*
joint log-density — priors, couplings **and** surrogate likelihoods. On this spec it
refuses to run, for an honest reason the next cell makes you look at.

In [ ]:
# `meta build` produced an IR: a flat, backend-neutral list of factors. This list *is* the
# model. `compiler.py::evaluate_log_prob` walks it to get a log-density, and under
# propagate `sampling.py::_sample_core` applies its coupling factors in exactly this order.
from bayesian_metamodeling.meta import build_ir_from_metamodel_spec

ir = build_ir_from_metamodel_spec(spec)
print(f"IR '{ir.name}': {len(ir.variables)} variables, {len(ir.factors)} factors\n")
for factor in ir.factors:
    if factor.kind == "prior":
        print(f"  prior                 {factor.variable:>2} ~ {factor.distribution}")
    elif factor.kind == "coupling":
        width = "exact" if factor.coupling_type == "deterministic_transform" else f"σ={factor.sigma}"
        print(
            f"  coupling              {factor.source:>2} -> {factor.target:<2} "
            f"{factor.coupling_type:<24} {width}"
        )
    else:
        print(
            f"  surrogate_likelihood  {str(factor.inputs):<6} -> {str(factor.outputs):<6} "
            f"ref={factor.surrogate_ref:<12} (INERT under propagate — no fitted model)"
        )

# Why "inert" — open one of the artifacts the spec references.
stub = json.loads((root / "tutorials/artifacts/surrogate_C.artifact.json").read_text())
print("\ntutorials/artifacts/surrogate_C.artifact.json:")
print(json.dumps(stub, indent=2))
print("\nKeys present:", sorted(stub), "-> no 'backend_payload', so no fitted model.")
print("It declares a signature (z -> w) so `meta build` can assemble the IR, and nothing")
print("more. `x` is in the sampled dataset for the same reason: it is surrogate_A's")
print("declared input, and no cell in this notebook has any use for it.")

# And this is what happens when you ask for the method that DOES need the surrogates.
print()
rc = run_mm_cli(
    'meta', 'sample', SPEC_REL,
    '--draws', '50', '--tune', '10', '--chains', '1', '--seed', '1', '--method', 'joint',
    check=False,
)
print(f"\nexit code {rc} — an honest refusal, not a bug. Joint sampling conditions on the")
print("surrogate likelihoods; there is nothing here to condition on. Propagation never")
print("noticed, because it never looked.")
assert rc != 0, "expected --method joint to refuse: this spec's surrogates are placeholders"

## Step 2: Read the noise budget off the samples (graphic)

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np

# --- Pick the dataset by PROVENANCE, not by guesswork --------------------------------
# Every stored sample records the sha256 of the spec that produced it. Recomputing that
# digest from the file on disk and matching it is how you know the numbers below belong to
# the spec in your editor. It is also what catches the classic mistake: edit a σ, forget to
# re-run `meta sample`, and read stale draws as if they were new.
def spec_digest(spec_obj) -> str:
    payload = json.dumps(spec_obj.model_dump(mode="json"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def load_samples(spec_obj, rerun_hint="the `meta sample` cell in Step 1"):
    """Newest stored dataset whose recorded spec_digest matches `spec_obj`."""
    registry = json.loads((root / "tmp/metamodel_samples_registry.json").read_text())
    want = spec_digest(spec_obj)
    hits = sorted(
        (meta.get("created_at", ""), meta)
        for meta in registry.values()
        if meta.get("spec_digest") == want
    )
    for _created, meta in reversed(hits):
        path = Path(meta["samples_dataset_path"])
        if not path.is_absolute():
            path = root / path
        if path.exists():
            variables = json.loads(path.read_text())["variables"]
            arrays = {k: np.asarray(v, dtype=float).reshape(-1) for k, v in variables.items()}
            return arrays, path
    raise RuntimeError(
        f"No stored samples match spec '{spec_obj.name}' (digest {want[:12]}...). "
        f"Re-run {rerun_hint}."
    )


samples, ds_path = load_samples(spec)
n_draws = len(samples["y"])
print(f"dataset: {ds_path.relative_to(root)}  ({n_draws} draws, chains flattened)\n")

# --- What the spec predicts, computed from the spec -----------------------------------
sigma = {(c.source, c.target): c.sigma for c in spec.couplings}
s1 = float(sigma[("y", "C")])
s2 = float(sigma[("z", "w")])
prior_sd = float(next(p.distribution["scale"] for p in spec.priors if p.variable == "y"))
predicted = {
    "y": prior_sd,
    "C": np.sqrt(prior_sd**2 + s1**2),
    "z": np.sqrt(prior_sd**2 + s1**2),
    "w": np.sqrt(prior_sd**2 + s1**2 + s2**2),
}

print("Noise budget along y → C → z → w. 'predicted' is quadrature on the spec's own σ")
print("values; 'observed' is the draws; 'discrepancy' is their difference in units of the")
print("Monte Carlo error, which for a std from N iid draws is std/sqrt(2N).\n")
print(f"{'var':<4} {'observed std':>13} {'mc error':>10} {'predicted':>11} {'discrepancy':>14}")
for name in ("y", "C", "z", "w"):
    observed = float(samples[name].std())
    mcse = observed / np.sqrt(2 * n_draws)
    print(
        f"{name:<4} {observed:13.4f} {mcse:10.4f} {predicted[name]:11.4f} "
        f"{(observed - predicted[name]) / mcse:+11.2f} se"
    )

print(
    f"\nstd(w)/std(y) = {samples['w'].std() / samples['y'].std():.3f}"
    f"    (predicted {predicted['w'] / predicted['y']:.3f})"
)
print(
    f"max |z - C|   = {np.abs(samples['z'] - samples['C']).max():.3e}"
    "    <- 'deterministic' means EXACTLY equal, not approximately"
)
print(
    f"corr(z, w)    = {np.corrcoef(samples['z'], samples['w'])[0, 1]:.3f}"
    f"    (predicted std(z)/std(w) = {predicted['z'] / predicted['w']:.3f})"
)

# --- Violin plot: widths side by side --------------------------------------------------
panels = {
    "y\n(prior)": (samples["y"], "tab:blue"),
    f"C\n(= y + N(0,{s1}))": (samples["C"], "tab:green"),
    "z\n(= C, determ.)": (samples["z"], "tab:green"),
    f"w\n(= z + N(0,{s2}))": (samples["w"], "tab:red"),
}
fig, ax = plt.subplots(figsize=(8.5, 5))
parts = ax.violinplot(
    [arr for arr, _ in panels.values()], showmeans=False, showmedians=True, widths=0.8
)
for body, (_, color) in zip(parts["bodies"], panels.values()):
    body.set_facecolor(color)
    body.set_alpha(0.55)

top = max(arr.max() for arr, _ in panels.values()) + 0.45
for i, (arr, _) in enumerate(panels.values(), start=1):
    ax.text(i, top, f"std = {arr.std():.3f}", ha="center", fontsize=10, fontweight="bold")
ax.text(
    2.5, top - 0.75, "identical, not merely similar", ha="center", fontsize=9, style="italic"
)
ax.set_xticks(range(1, len(panels) + 1))
ax.set_xticklabels(list(panels))
ax.set_ylabel("value")
ax.set_title("Forward propagation along the chain  y → C → z → w   (--method propagate)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Step 3: The tell — propagation is order-sensitive, a joint density is not

Here is the sharpest available demonstration that `propagate` is not inference, and it
costs one re-run.

`_sample_core` applies coupling factors **in the order the spec lists them**, and each one
*overwrites* its target:

```python
samples[factor.target] = transform(samples[factor.source])   # (+ noise, if probabilistic)
```

So move `z → w` to the **front** of the `couplings` array. Nothing errors and nothing
warns — but `w` is now built from the *fallback prior draw* of `z` (the `Normal(0, 1)` the
first cell warned you about), which the later `C → z` link then overwrites. `w` ends up a
sibling of the chain instead of its descendant.

A joint density does not behave this way. `compiler.py::evaluate_log_prob` **sums** its
factors, and addition is commutative: reorder the same factors and you get the same
density, hence the same posterior. Order-sensitivity is the signature of a post-draw
transform chain; order-freedom is the signature of a joint.

Predict before you run: in the reordered version, what happens to `corr(z, w)`? And to
`std(w)` — up, down, or unchanged?

In [ ]:
# Same three couplings, same σ values, same seed, same draws — only the ORDER changes.
scrambled = json.loads(SPEC_PATH.read_text())
scrambled["name"] = "tutorial_three_model_coupling_scrambled"
couplings = scrambled["couplings"]
scrambled["couplings"] = [couplings[2], couplings[0], couplings[1]]  # z→w first

scrambled_rel = "tmp/tutorials/metamodel.three_model.scrambled.json"
scrambled_path = root / scrambled_rel
scrambled_path.parent.mkdir(parents=True, exist_ok=True)
scrambled_path.write_text(json.dumps(scrambled, indent=2))

_ = run_mm_cli(
    'meta', 'sample', scrambled_rel,
    '--draws', '4000', '--tune', '100', '--chains', '2', '--seed', '321',
)

scrambled_spec = MetaModelSpec.model_validate_json(scrambled_path.read_text())
reordered, _ = load_samples(scrambled_spec, rerun_hint="this cell")


def summarise(tag, s):
    print(
        f"{tag:<26} std(y)={s['y'].std():.3f}  std(z)={s['z'].std():.3f}  "
        f"std(w)={s['w'].std():.3f}  corr(z,w)={np.corrcoef(s['z'], s['w'])[0, 1]:+.3f}"
    )


print()
summarise("spec order  (y→C→z→w)", samples)
summarise("z→w moved to the front", reordered)

corr_ok = float(np.corrcoef(samples["z"], samples["w"])[0, 1])
corr_bad = float(np.corrcoef(reordered["z"], reordered["w"])[0, 1])
assert corr_ok > 0.4 and abs(corr_bad) < 0.15, (
    f"expected corr(z,w) high in spec order and ~0 reordered, got {corr_ok:.3f} and "
    f"{corr_bad:.3f}. If both are high, coupling application is no longer order-sensitive "
    "and this lesson needs rewriting."
)

print(
    "\nw is still wide, still plausible, still passes every smell test — and it is now"
    "\nstatistically independent of the chain it is supposed to descend from. Nothing"
    "\nraised; nothing warned; the spec still says y → C → z → w."
)
print(
    f"\nNote which statistic caught it. A width check sees std(w)="
    f"{reordered['w'].std():.3f} > std(y)={reordered['y'].std():.3f} and calls the run"
    "\nhealthy. Only the correlation notices that the chain has been severed — which is"
    "\nwhy the self-check at the bottom asserts a correlation and not a width."
)

## Scientific checkpoint (active learning)

1. **Did quadrature hold?** Read the `discrepancy` column, not the raw numbers. Each entry
   is (observed − predicted) in units of the Monte Carlo error *on that measurement*.
   Anything inside about ±2 se is agreement. A gap of a hundredth means nothing when the
   error bar is a hundredth too — and you cannot know that without printing the error bar.
   Take the habit away with you: **every sampled number carries one.**

2. **Predict, then test.** Double σ on the `z → w` link (0.8 → 1.6) in
   `tutorials/specs/metamodel.three_model.pymc.json`, save, and re-run Steps 1-3. Predicted:
   `std(w)` moves from `sqrt(1 + 0.36 + 0.64) = 1.414` to `sqrt(1 + 0.36 + 2.56) = 1.980` —
   a factor of 1.40, not the factor of 2 you applied to σ. Quadrature is sublinear, and the
   prior variance you are adding to does not go away. Then try σ = 0.1: `w` becomes almost
   indistinguishable from `z`, which is the regime where a "noticeable" effect is really
   just Monte Carlo noise. The self-check at the bottom reads σ **from the spec**, so it
   re-derives the prediction and still passes: the numbers move, the law does not.

3. **The deterministic step is free.** `max |z − C|` printed as exactly `0.000e+00`, and
   `std(z) = std(C)` to the last digit. A deterministic coupling transmits width; it does
   not create any. Two design consequences: a hard constraint you can justify costs nothing
   in the noise budget, and `z` tells you nothing that `C` had not already said.

4. **What none of this shows.** Every number above came from priors and transforms. No
   surrogate was evaluated, no data was conditioned on, and `y` still sits at exactly its
   prior. If you want three models to *inform each other* rather than merely feed each
   other, you need `--method joint` and real fitted surrogates.

5. **Where the lesson lands for real models.** Open
   `projects/tcr_signaling/specs/metamodel.tcr_signaling.json`. It couples four partial
   models and declares exactly **one** coupling — a `deterministic` affine link from
   `contact_fraction` to `cd45_boundary_density`. By this tutorial's own logic that link
   contributes zero width, so `propagate` on it would be almost content-free: the
   uncertainty in that project lives in the four *fitted surrogate likelihoods*, which
   propagation never evaluates. That is exactly why it runs `--method joint` (see
   `projects/tcr_signaling/notebooks/03_metamodel_inference.ipynb`). A cascade of σ's like
   this notebook's is the teaching case; the real one is a single hard link plus four
   likelihoods.

**Reset σ on the `z → w` link to 0.8** when you are done, so later runs match what this
notebook describes.

## Troubleshooting

| Symptom | Cause | What to do |
|---|---|---|
| `Joint sampling unavailable: ... has no 'backend_payload'` | The artifacts this spec names are placeholders — a declared signature, no fitted model | Not a bug; Step 1 triggers it on purpose. Joint sampling conditions on the surrogate likelihoods, so it needs real fits (T5/T6 produce them; `projects/tcr_signaling` notebook 03 runs the real thing). `propagate` never notices because it never evaluates them. |
| `std(w)` barely differs from `std(y)` | σ is small next to the prior width, and widths add in quadrature | Work in variances. σ = 0.25 against a prior sd of 1.0 buys `sqrt(1.0625) = 1.031` — 3%, which is below the Monte Carlo error of a few-hundred-draw run. Raise σ toward the prior width if you want to *see* the effect. |
| Numbers shift when you change `--seed` | Monte Carlo error; every sampled statistic has one | Judge discrepancies in units of `mc error`, never in absolute terms. The error falls as `1/sqrt(N)`, so 4× the draws halves it. |
| A variable is silently detached from the chain — no error, plausible width | Under `propagate` couplings are post-draw overwrites applied in spec order; a target built before its source is computed gets the fallback prior draw | List couplings source-first (Step 3 is this failure, staged), or use `--method joint`, whose log-density is a sum and therefore order-free. |
| Adding a third model changes nothing downstream | Its coupling target is overwritten by a later coupling, or it enters only through a surrogate likelihood — inert under `propagate` | Print the IR factor list (Step 1) and check where the variable actually appears, rather than assuming the spec's intent reached the sampler. |

**Next:** T9 hands you the keys. You'll compose spec → sweep → surrogate → evaluation
yourself, without step-by-step instructions, on a model you choose.

## Final check: T8's three claims, re-derived from the stored draws

The check below re-loads both datasets **by spec digest**, so it cannot pass on a stale
file, and asserts the three things this notebook actually taught:

1. **Determinism is exact** — `max |z − C| = 0`, not "small". If the deterministic coupling
   ever injected noise, this fails.
2. **Widths follow the spec's σ values in quadrature** — observed variances for `y`, `z`
   and `w` against `prior² + Σσ²` computed from the spec, required within 5 Monte Carlo
   standard errors. It reads σ from the spec, so it stays honest after the checkpoint
   exercise. A dropped coupling, a σ that never reached the sampler, or a stale dataset all
   fail here.
3. **Order-sensitivity is real** — `corr(z, w)` is high in spec order and ~0 in the
   reordered run. This fails if Step 3 never ran, and it is precisely the claim a width
   comparison cannot make: in the reordered run `std(w)` is still comfortably larger than
   `std(y)`, so a "did the width grow?" test passes on a chain that is broken.

In [ ]:
# Self-check: T8's claims, each re-derived from the stored datasets rather than from any
# variable this notebook happens to have in memory.
import hashlib as _hashlib
import json as _json
from pathlib import Path as _P

import numpy as _np

from bayesian_metamodeling.spec import MetaModelSpec as _Spec

_spec_path = root / "tutorials/specs/metamodel.three_model.pymc.json"
_scrambled_path = root / "tmp/tutorials/metamodel.three_model.scrambled.json"


def _digest(_sp) -> str:
    return _hashlib.sha256(
        _json.dumps(_sp.model_dump(mode="json"), sort_keys=True).encode("utf-8")
    ).hexdigest()


def _load(_sp, _hint):
    _reg = _json.loads((root / "tmp/metamodel_samples_registry.json").read_text())
    _want = _digest(_sp)
    _hits = sorted(
        (_m.get("created_at", ""), _m) for _m in _reg.values() if _m.get("spec_digest") == _want
    )
    for _created, _m in reversed(_hits):
        _p = _P(_m["samples_dataset_path"])
        if not _p.is_absolute():
            _p = root / _p
        if _p.exists():
            _v = _json.loads(_p.read_text())["variables"]
            return {k: _np.asarray(a, dtype=float).reshape(-1) for k, a in _v.items()}
    raise AssertionError(
        f"no stored samples match spec '{_sp.name}' (digest {_want[:12]}...) — re-run {_hint}"
    )


_spec = _Spec.model_validate_json(_spec_path.read_text())
_s = _load(_spec, "Step 1 (`meta build` + `meta sample`)")
_y, _C, _z, _w = _s["y"], _s["C"], _s["z"], _s["w"]
_n = len(_y)

# (1) `deterministic` means exactly equal, not approximately equal.
_gap = float(_np.abs(_z - _C).max())
assert _gap <= 1e-12, (
    f"max|z - C| = {_gap:.3e}, expected 0. The C -> z coupling is `deterministic`, which "
    "asserts z = 1.0*C + 0.0 exactly; any spread means the deterministic primitive is "
    "injecting noise it should not."
)

# (2) Widths follow the spec's σ values in quadrature, judged in units of MC error.
#     The sd of a sample variance from N iid normal draws is var*sqrt(2/N).
_sig = {(c.source, c.target): float(c.sigma) for c in _spec.couplings if c.sigma is not None}
_prior_var = float(next(p.distribution["scale"] for p in _spec.priors if p.variable == "y")) ** 2
_expected_var = {
    "y": _prior_var,
    "z": _prior_var + _sig[("y", "C")] ** 2,
    "w": _prior_var + _sig[("y", "C")] ** 2 + _sig[("z", "w")] ** 2,
}
_dev = {}
for _name, _var_pred in _expected_var.items():
    _var_obs = float(_s[_name].var())
    _dev[_name] = (_var_obs - _var_pred) / (_var_pred * _np.sqrt(2.0 / _n))
    assert abs(_dev[_name]) < 5.0, (
        f"var({_name}) = {_var_obs:.4f} but the spec predicts {_var_pred:.4f} "
        f"({_dev[_name]:+.1f} Monte Carlo standard errors away). Quadrature along "
        "y -> C -> z -> w says var = prior_var + the sum of the coupling sigmas squared. "
        "If you edited a sigma, re-run Step 1 so the draws match the spec on disk; "
        "otherwise a coupling is not doing what the spec says it does."
    )

# (3) Order-sensitivity: the reordered run detaches w from the chain.
assert _scrambled_path.exists(), "Step 3 never ran — the reordered spec was never written."
_bad = _load(_Spec.model_validate_json(_scrambled_path.read_text()), "Step 3")
_corr_ok = float(_np.corrcoef(_z, _w)[0, 1])
_corr_bad = float(_np.corrcoef(_bad["z"], _bad["w"])[0, 1])
assert _corr_ok > 0.4 and abs(_corr_bad) < 0.15, (
    f"corr(z, w) = {_corr_ok:.3f} in spec order and {_corr_bad:.3f} with z->w moved to the "
    "front. Expected high, then ~0: under --method propagate a coupling is a post-draw "
    "overwrite, so listing one before its source is computed silently detaches the target "
    "from the chain."
)

print(
    f"\n[T8 self-check OK] max|z-C|={_gap:.1e} (exact); "
    f"var(y),var(z),var(w) within {max(abs(v) for v in _dev.values()):.1f} MC se of the "
    f"spec's quadrature prediction; corr(z,w)={_corr_ok:.3f} in spec order vs "
    f"{_corr_bad:+.3f} reordered."
)
print(
    f"  (width-only comparison on the reordered run: std(w)={_bad['w'].std():.3f}, "
    f"std(y)={_bad['y'].std():.3f} — which is why claim 3 is a correlation, not a width.)"
)